# Attacks on the dataset belonging to the paper "Online Template attacks"

In [ ]:
import trsfile
import matplotlib.pyplot as plt
import numpy as np
from tqdm import trange, tqdm
import seaborn as sns
import bokeh.plotting as bk
import lpips
import librosa
import torchaudio
import torch
import math

from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn.functional as F
import numpy as np

from bokeh.io import output_notebook 
output_notebook()

## Loading the dataset

In [ ]:
## Open the dataset
with trsfile.open('../../datasets/ota1/attack + affine_templates.trs', 'r') as traces:
    # Show all headers

    for header, value in traces.get_headers().items():
        print(header, '=', value)

        if str(header) == "Header.NUMBER_TRACES":
            nr_traces = value
        elif str(header) == "Header.NUMBER_SAMPLES":
            nr_samples = value
    print()

    trace_array = np.zeros((nr_traces, nr_samples))

    # Iterate over the first 11 traces
    for i, trace in enumerate(traces):
        print('Trace {0:d} contains {1:d} samples'.format(i, len(trace)))
        print('  - minimum value in trace: {0:f}'.format(min(trace)))
        print('  - maximum value in trace: {0:f}'.format(max(trace)))
        trace_array[i, :] = trace



sample_rate = 125e6 

# Split the target trace from the template traces
target_trace = trace_array[0, :]
raw_template_traces = trace_array[1:, :]

print(target_trace.shape) 

### Helper functions

In [3]:
# Function to cut off the "0 part" in template traces
def slice_trace(trace):
    idx = np.where(trace == 0)[0][0] # Get the first index where the power consumption = 0 [note, might not work for all datasets]
    new_trace = trace[:idx]
    return new_trace

def compute_correlation(target_trace, template_trace):
    corr = np.corrcoef(target_trace, template_trace)[0,1] # Take the non-identity part of the correlation matrix. Note that [0,1] = [1,0]
    return np.abs(corr)

def pattern_match(target_trace, template_trace, comparison_function, poi_start, poi_end, step=1):
    comparison_array = []
    window = poi_end - poi_start
    template_trace_sliced = template_trace[poi_start:poi_end]
    for timesample in trange(0, len(target_trace) - window, step):
        comparison = comparison_function(target_trace[timesample:timesample + window], template_trace_sliced)
        comparison_array.append(comparison)
    return np.array(comparison_array)

def determine_keybit(comparison_array0, comparison_array1, sign):
    # Get the highest peak of both, and take that timesample as comparison point
    if sign == "max":
        peak0 = np.max(comparison_array0)
        peak1 = np.max(comparison_array1)
        keybit_guess = np.argmax([peak0, peak1]) 
        print("Score bit 0: ", peak0)
        print("Score bit 1: ", peak1)
        timesample = np.argmax(comparison_array0) if keybit_guess == 0 else np.argmax(comparison_array1)
    elif sign == "min":
        peak0 = np.min(comparison_array0)
        peak1 = np.min(comparison_array1)
        keybit_guess = np.argmin([peak0, peak1]) 
        print("Score bit 0: ", peak0)
        print("Score bit 1: ", peak1)
        timesample = np.argmin(comparison_array0) if keybit_guess == 0 else np.argmin(comparison_array1)
    
    return int(keybit_guess), timesample

## Visualisation of the dataset

In [ ]:
# Plot the target trace. Use block zoom to zoom in on the traces (square with magnifier in the plot)
p = bk.figure(title="Target trace", x_axis_label='Timesample', y_axis_label='Delta power consumption', width=800, height=400)
p.line(np.arange(0, len(target_trace)), target_trace, line_width=2)
bk.show(p)

# Plot the first template trace
p = bk.figure(title="Raw template trace 2P", x_axis_label='Timesample', y_axis_label='Delta power consumption', width=800, height=400)
p.line(np.arange(0, len(raw_template_traces[0,:])), raw_template_traces[0,:], line_width=2)
bk.show(p)

# Cut off all 0 for each template trace
template_traces = np.zeros((10, np.where(raw_template_traces[0] == 0)[0][0]))

for i, trace in enumerate(raw_template_traces):
    new_trace = slice_trace(trace)
    template_traces[i] = new_trace

print(template_traces.shape)

# Plot the new template traces
p = bk.figure(title="Template trace 2P", x_axis_label='Timesample', y_axis_label='Delta power consumption', width=800, height=400)
p.line(np.arange(0, len(template_traces[0,:])), template_traces[0,:], line_width=2)
bk.show(p)

# Overlay the two traces. Note that they are not aligned. 
p = bk.figure(title="Template trace 2P + Target trace", x_axis_label='Timesample', y_axis_label='Delta power consumption', width=800, height=400)
p.line(np.arange(0, len(target_trace[:1000])), target_trace[:1000], line_width=2)
p.line(np.arange(0, len(template_traces[0,600:1600])), template_traces[0,600:1600], line_width=2, color="red")
bk.show(p)

## Correlation based attack - reproduced OTA


In [ ]:
# Do the sequential key bit recovery
nr_keybits = 5
step_size = 1
poi_start = 2024  # This window determined by pattern identification, such as e.g. autocorrelation (used in the original OTA paper)
poi_end = 8622
window = poi_end - poi_start
valleys = [0, 135000, 312000, 490000, 670000, 810000] # using SPA
scalar = 1

corr_array_size = math.ceil((nr_samples-window) / step_size)

corr0 = np.zeros((nr_keybits, corr_array_size))
corr1 = np.zeros((nr_keybits, corr_array_size))

recovered_key = [1] # MSB is always 1, add it already to the array

for iteration, keybit_to_guess in enumerate(range(0, nr_keybits*2, 2)):

    # Determine input points based on previous key bits
    # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
    
    # Collect traces
    # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

    # Create templates from the collected traces
    template0 = template_traces[keybit_to_guess, :]
    template1 = template_traces[keybit_to_guess+1, :]

    # Perform pattern matching with templates to target trace based on Correlation
    corr_results_template0 = pattern_match(target_trace, template0, compute_correlation, poi_start, poi_end, step=step_size)
    corr_results_template1 = pattern_match(target_trace, template1, compute_correlation, poi_start, poi_end, step=step_size)

    np.savez("./ota1_bitpos{}_{}.npz".format(iteration, compute_correlation.__name__), comparison_0 = corr_results_template0, comparison_1 = corr_results_template1)

    # Optional: save for visualisations later
    corr0[iteration] = corr_results_template0 
    corr1[iteration] = corr_results_template1 

    keybit_guess, _ = determine_keybit(corr_results_template0, corr_results_template1, "max")

    scalar = (scalar << 1) | keybit_guess
    print("Current scalar value: " + str(scalar))

    recovered_key.append(keybit_guess)


print("The found key: " + str(recovered_key))



In [ ]:
# Plot the correlation graphs
titles = ["2P vs 3P", "4P vs 5P", "8P vs 9P", "18P vs 19P", "38P vs 39P"]

for i in range(0,5):
    plt.figure(figsize=(10,5))
    plt.plot(corr0[i ], label = "bit 0")
    plt.plot(corr1[i], label = "bit 1")
    plt.title("Correlation Template for {}".format(titles[i]))
    plt.xlabel("Timesamples")
    plt.ylabel("Delta power consumption")
    plt.legend()
    plt.savefig("./ota1_bitpos{}_computecorr.png".format(i), bbox_inches='tight')
    plt.show()

## Similarity based Attacks (SSIM + LPIPS)

In [12]:
def calc_spectrogram(trace, n_fft = 512): #NOTE: it is very sensitive for this value
    trace_tensor = torch.tensor(trace, dtype=torch.float32)
    spec = torchaudio.transforms.Spectrogram(n_fft=n_fft)(trace_tensor)
    spec_db = librosa.power_to_db(spec.numpy(), ref=np.max)

    # Normalize to [0,1]
    spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)

    return spec_db

def spec_to_tensor(spec_db, target_size):
    # Convert to 3-channel float32 tensor for LPIPS: (C, H, W)
    spec_tensor = torch.tensor(spec_db).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    # Resize to target_size
    spec_resized = F.interpolate(spec_tensor, size=target_size, mode='bilinear', align_corners=False)

    return spec_resized

In [ ]:
## LPIPS model
device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_model = lpips.LPIPS(net='alex')
lpips_model.eval()
lpips_model.to(device)


In [14]:
def compute_distance_lpips(target_trace, template_trace):
    tensor_size = (224, 224)  # LPIPS expects 224x224 input
    
    target_spec  = calc_spectrogram(target_trace, 512)
    template_spec = calc_spectrogram(template_trace, 512)

    target_tensor = spec_to_tensor(target_spec, tensor_size).to(device)
    template_tensor = spec_to_tensor(template_spec, tensor_size).to(device)

    d = lpips_model.forward(target_tensor,template_tensor)
    return d.detach().cpu().numpy().ravel()[0]

## Structural similarity (SSIM)
def compute_structural_similarity(target_trace, template_trace):
    """Compute structural similarity between two traces."""
    target_tensor = calc_spectrogram(target_trace, 512)
    template_tensor = calc_spectrogram(template_trace, 512)

    return ssim(target_tensor, template_tensor, data_range=target_tensor.max() - target_tensor.min())

In [ ]:
trace1 = np.random.randn(nr_samples).astype(np.float32)
trace2 = np.random.randn(nr_samples).astype(np.float32)

similarity = compute_structural_similarity(trace1, trace2)
print(f"Structural Similarity (SSIM): {similarity:.4f}")

distance = compute_distance_lpips(trace1, trace2)
print(f"LPIPS distance: {distance:.4f}")

In [16]:
def plot_similarity(similarity_array0, similarity_array1, scalars, left, right, similarity_function):
        plt.figure(figsize=(10,5))
        plt.plot(similarity_array0, label = "bit 0")
        plt.plot(similarity_array1, label = "bit 1")
        plt.axvspan(left, right, alpha=0.2)
        plt.title("Similarity for template for {} on {}".format(scalars, similarity_function.__name__))
        plt.xlabel("Timesamples")
        plt.ylabel("Delta power consumption")
        plt.legend()
        plt.savefig("./ota1_bitpos{}_{}.png".format(iteration, similarity_function.__name__), bbox_inches='tight')
        plt.show()

In [ ]:
# Do the sequential key bit recovery
keybits = 5
step_size = 1

similarity_methods = [compute_distance_lpips, compute_structural_similarity]
scalars = ["2P vs 3P", "4P vs 5P", "8P vs 9P", "18P vs 19P", "38P vs 39P"]

poi_start = 2024  # This window determined by pattern identification, such as e.g. autocorrelation (used in the original OTA paper)
poi_end = 8622
window = poi_end - poi_start

recovered_key = np.ones(shape=(len(similarity_methods), keybits + 1))  # Initialize with ones to set the first bit for each method to 1

valleys = [0, 135000, 312000, 490000, 670000, 810000] # using SPA to speed up computation time

# Perform pattern matching based on different similarity methods
for j, method in enumerate(similarity_methods):
    scalar = 1
    for iteration, keybit_to_guess in enumerate(range(0, keybits*2, 2)):
        # Determine input points based on previous key bits
        # [It is an online attack, but as the dataset is already there, assume that new input points are calculated here]
        
        # Collect traces
        # [It is an online attack, but as the dataset is already there, assume that new input points are put in the code & traces are collected]

        # Create templates from the collected traces
        template0 = template_traces[keybit_to_guess, :]
        template1 = template_traces[keybit_to_guess+1, :]

        # Select current window to attack based on SPA. Only done to speed up computation time
        start_window = valleys[iteration]
        end_window = valleys[iteration + 1]

        # Slice target trace based on the window where we attack
        target_slice = target_trace[start_window:end_window]

        similarity_array0 = pattern_match(target_slice, template0, method, poi_start, poi_end, step=step_size)
        similarity_array1 = pattern_match(target_slice, template1, method, poi_start, poi_end, step=step_size)

        np.savez("./ota1_bitpos{}_{}.npz".format(iteration, method.__name__), comparison_0 = similarity_array0, comparison_1 = similarity_array1)

        best_keyguess, timesample  = determine_keybit(similarity_array0, similarity_array1, sign="max" if method != compute_distance_lpips else "min")

        plot_similarity(similarity_array0, similarity_array1, scalars[iteration], timesample-5000, timesample+5000, method)

        recovered_key[j, iteration+1] = best_keyguess
        scalar = (scalar << 1) | best_keyguess
        print("Current scalar value: " + str(scalar))
        print("Recovered keybit: ", best_keyguess)

print("The correct key: 100110")

for nr, sim_method in enumerate(similarity_methods):
    print(f"Recovered key using {sim_method.__name__}: " + str(recovered_key[nr]))